# Validating the Full Lower Bound (Section 4.4)

This notebook validates, on small examples, the argument behind **Theorem 4.17** of the
paper (*Chromatic word-quasisymmetric functions of matroids*): the chromatic
word-quasisymmetric functions $\{\Psi(N)\}$ of *all* $d!$ loopless nested matroids on
$[d]$ are linearly independent, so $\dim_{\mathbb Q}(\operatorname{im}\Psi)_d = d!$
exactly, matching the upper bound and (Corollary 4.28) fully resolving Conjecture 1.3.

A first complete draft of this argument (Section 4.4, Lemmas 4.19-4.24, Definition 4.25,
Proposition 4.26) was produced by the AI model **Claude Fable 5.1**, then verified,
partially re-proved, and rewritten by the authors before being included in the paper —
Claude is credited as a coauthor for this contribution; see the paper's *Disclosure of AI
assistance*. This notebook independently checks the individual claims of that argument by
direct computation on small matroids — a good way to build confidence in (and intuition
for) a proof that is otherwise fairly abstract.

This is a *different* result from the one tested in `03_min_max_conjecture_testing.ipynb`.
That notebook tests **Conjecture 4.14** (the *min-max* set compositions), which is still
open: it asks for linear independence witnessed by one *specific* family of $d!$ set
compositions. The proof validated here instead uses a family of set compositions built
from *complete flags of flats* (Lemma 4.24), which is provably enough to get linear
independence (Theorem 4.17) — but this does **not** establish Conjecture 4.14: the
flag-of-flats family and the min-max family are different in general (the flag-of-flats
witnesses vary by matroid rank via Hampe's theorem, rather than being one fixed family of
$d!$ set compositions), so the min-max conjecture remains a separate open question.

We reuse the `chromatic_matroids` package for matroids, set compositions and the
$\Psi$-genericity check (`stable_matroids_setcompositions`). The matroid operations
Section 4.4 needs beyond `rank(subset)` — `flats`, `cyclic_flats`, `loops`, `restriction`,
`contraction` and `minor` — were added to the `Matroid` class as part of writing this
notebook (they did not exist before); see the next section for why, and for a genuine bug
in the old `is_nested()` that turned up along the way and has now been fixed in the
package.

In [1]:
# Install the package from local sources (run once per environment)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "../package", "--quiet"])

0

In [2]:
import itertools
import time
from math import factorial

import numpy as np

from chromatic_matroids import (
    Matroid,
    SetComposition,
    schubert_matroid,
    uniform_matroid,
    generate_loopless_nested_matroids,
    stable_matroids_setcompositions,
    z_rank,
)

print("✓ All imports successful!")

✓ All imports successful!


## Matroid operations: flats, cyclic flats, loops, restriction, contraction, minor

The `Matroid` class used to only provide `rank(subset)`. Section 4.4 needs flats, loops,
and the two matroid minors (restriction and contraction), so these are now `Matroid`
methods, built directly from the rank function exactly as in the paper's Preliminaries and
Section 4.4: `M.flats()`, `M.cyclic_flats()`, `M.loops()`, `M.restriction(F)`,
`M.contraction(F)`, and the convenience `M.minor(Fi, Fim1)` for `(M|Fi)/Fim1`.

Two subtleties turned up while writing them, one of which was a real bug, now fixed:

* **Restriction is *not* simply `{B ∩ F : B a basis}`.** For $B$ a basis and $F$ a subset,
  the sets $B \cap F$ do *not* all have the same size in general (e.g. take $M = U_{1,2}$
  on $\{1,2\}$ and $F = \{1\}$: the basis $\{1\}$ gives $B\cap F = \{1\}$, but the basis
  $\{2\}$ gives $B\cap F = \emptyset$). The bases of $M|_F$ are exactly the intersections
  of *maximum* size, $\operatorname{rk}_M(F)$; `Matroid.restriction` filters for this.
* **`Matroid.is_nested()` used to check a different (and in general false) condition.** It
  tested whether *all* flats of $M$ form a chain, but nested matroids (as defined in the
  paper, and used here) only require their *cyclic* flats to form a chain — the full
  lattice of flats need not be a chain. Indeed `SM({2,4})` on $[4]$ below has incomparable
  flats $\{1\}$ and $\{2\}$, even though it is nested (its *cyclic* flats are
  $\emptyset \subsetneq \{3,4\} \subsetneq [4]$, a chain). This is exactly why Lemma 4.24
  (a *complete* flag of flats reading off the rank-one pieces) is a genuine, non-trivial
  statement about nested matroids, rather than an immediate consequence of nestedness.
  `Matroid.is_nested()` has been fixed to use `Matroid.cyclic_flats()`, and the (now
  corrected) `Matroid.flats()` is kept as its own useful method.

In [3]:
# quick demo: the two matters above, on the example used throughout this notebook
M_demo = schubert_matroid(4, frozenset([2, 4]))
print("SM({2,4}) on [4]:")
print("  bases        =", sorted(tuple(sorted(b)) for b in M_demo.bases_sets))
print("  loops        =", M_demo.loops())
print("  flats        =", sorted(tuple(sorted(f)) for f in M_demo.flats()))
print("  cyclic_flats =", sorted(tuple(sorted(f)) for f in M_demo.cyclic_flats()))
print("  is_nested()  =", M_demo.is_nested(), " <- now correctly True (fixed bug: used to be False)")

SM({2,4}) on [4]:
  bases        = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4)]
  loops        = frozenset()
  flats        = [(), (1,), (1, 2, 3, 4), (2,), (3, 4)]
  cyclic_flats = [(), (1, 2, 3, 4), (3, 4)]
  is_nested()  = True  <- now correctly True (fixed bug: used to be False)


## Antigenericity and the block-reversal trick (Lemma 4.19)

Section 4.4 works with the auxiliary *antigeneric* notion (unique **minimum**-weight
basis) rather than $\Psi$-genericity (unique **maximum**-weight basis) directly, because
that is what makes the "minimal elements of the exchange poset" combinatorics come out to
`Loops(N)` rather than the coloops. Concretely, $\pi$ is $N$-antigeneric iff its
block-reversal $\pi^{\mathrm{rev}}$ is $N$-generic in the usual (package) sense, since
reversing the block order negates the weight function $f_\pi$.

We also define the coefficient $a(\pi) = (-1)^{|S|-\ell(\pi)}\,|\pi_1|/|S|$ used
throughout, and the local rank-one detector $\delta_S(N) = \sum_\pi a(\pi)\,g_N(\pi)$ from
Lemma 4.22, which Lemma 4.22 says should equal $\mathbb 1\{\operatorname{rk}(N) +
|\mathrm{Loops}(N)| = 1\}$.

In [4]:
def reverse_composition(opi: SetComposition) -> SetComposition:
    '''pi_1|...|pi_t  |->  pi_t|...|pi_1'''
    return SetComposition(list(reversed(opi.parts)))


def is_generic(M: Matroid, opi: SetComposition) -> bool:
    return stable_matroids_setcompositions(M, opi)


def is_antigeneric(M: Matroid, opi: SetComposition) -> bool:
    return is_generic(M, reverse_composition(opi))


def a_coeff(opi: SetComposition) -> float:
    S = len(opi.ground_set)
    t = len(opi.parts)
    first = len(opi.parts[0])
    return ((-1) ** (S - t)) * first / S


def all_setcompositions_on(elements) -> list:
    '''All set compositions of a (not necessarily {1,...,n}) finite set of labels.'''
    elements = sorted(elements)
    n = len(elements)
    if n == 0:
        return [SetComposition()]
    relabel = {i + 1: elements[i] for i in range(n)}
    return [sc.relabel(relabel) for sc in SetComposition.generate_all_setcompositions(n)]


def delta_S(M: Matroid) -> float:
    '''Lemma 4.22's local rank-one detector delta_S(N).'''
    if len(M.ground_set) == 0:
        return 0.0
    return sum(a_coeff(opi) for opi in all_setcompositions_on(M.ground_set) if is_antigeneric(M, opi))


print("a((1,2|3)) =", a_coeff(SetComposition([[1, 2], [3]])))

a((1,2|3)) = -0.6666666666666666


## Validating Lemma 4.20 (exchange posets)

For a matroid $N$ on $S$ and a basis $B$, Lemma 4.20(i) claims: $B$ is the *unique*
minimum-$f_\pi$-weight basis if and only if $f_\pi$ is strictly order preserving on the
exchange poset $P_B$ (i.e. $f_\pi(b) < f_\pi(c)$ for every covering relation $b <_{P_B} c$,
where $b \in B$, $c \notin B$, $(B\setminus b)\cup c$ a basis).

We check this by brute force, for *every* set composition $\pi$ of $[4]$ and *every*
basis $B$ of `SM({2,4})`, against the actual unique-minimum-basis computation.

In [5]:
def f_weight(opi: SetComposition, e: int) -> int:
    for idx, part in enumerate(opi.parts):
        if e in part:
            return idx
    raise ValueError(f"{e} not in {opi}")


def min_weight_bases(M: Matroid, opi: SetComposition) -> list:
    scores = {B: sum(f_weight(opi, e) for e in B) for B in M.bases_sets}
    m = min(scores.values())
    return [B for B, s in scores.items() if s == m]


def exchange_poset_relations(M: Matroid, B) -> list:
    '''pairs (b, c) with b in B, c not in B, (B - b) + c a basis of M.'''
    return [
        (b, c)
        for b in B
        for c in M.ground_set - B
        if frozenset((B - {b}) | {c}) in M.bases_sets
    ]


def order_preserving(opi: SetComposition, relations: list) -> bool:
    return all(f_weight(opi, b) < f_weight(opi, c) for (b, c) in relations)


M = schubert_matroid(4, frozenset([2, 4]))
checked, mismatches = 0, 0
for opi in all_setcompositions_on(M.ground_set):
    mwb = min_weight_bases(M, opi)
    for B in M.bases_sets:
        predicted = order_preserving(opi, exchange_poset_relations(M, B))
        actual = (len(mwb) == 1 and mwb[0] == B)
        checked += 1
        mismatches += (predicted != actual)

print(f"Checked {checked} (set composition, basis) pairs for SM({{2,4}}) on [4].")
print(f"Mismatches with Lemma 4.20(i): {mismatches}")
assert mismatches == 0

Checked 375 (set composition, basis) pairs for SM({2,4}) on [4].
Mismatches with Lemma 4.20(i): 0


## Validating Lemma 4.22 (the local rank-one detector)

$\delta_S(N)$ should equal $1$ exactly when $N$ is loopless of rank $1$, or has rank $0$
with $|S|=1$ (a single loop), and $0$ otherwise.

In [6]:
tests = [
    ("U(1,3)  [rank 1, loopless]",           uniform_matroid(3, 1)),
    ("U(2,3)  [rank 2, loopless]",           uniform_matroid(3, 2)),
    ("SM({2,4}) on [4]  [rank 2, loopless]", schubert_matroid(4, frozenset([2, 4]))),
    ("SM({1}) on [3]  [has loops]",          schubert_matroid(3, frozenset([1]))),
    ("SM({1,2}) on [3]  [loopless, rank 2]", schubert_matroid(3, frozenset([1, 2]))),
    ("{1} as a single loop  [rank 0]",       Matroid(frozenset([1]), {frozenset()})),
]

print(f"{'matroid':38s} {'rk':>3s} {'#loops':>7s} {'delta_S':>9s} {'predicted':>10s}  match")
for name, N in tests:
    rk = N.rank(N.ground_set)
    L = N.loops()
    predicted = 1.0 if (rk + len(L) == 1) else 0.0
    computed = delta_S(N)
    match = abs(computed - predicted) < 1e-9
    print(f"{name:38s} {rk:3d} {len(L):7d} {computed:9.4f} {predicted:10.1f}  {match}")
    assert match

matroid                                 rk  #loops   delta_S  predicted  match
U(1,3)  [rank 1, loopless]               1       0    1.0000        1.0  True
U(2,3)  [rank 2, loopless]               2       0    0.0000        0.0  True
SM({2,4}) on [4]  [rank 2, loopless]     2       0    0.0000        0.0  True
SM({1}) on [3]  [has loops]              1       2    0.0000        0.0  True
SM({1,2}) on [3]  [loopless, rank 2]     2       1    0.0000        0.0  True
{1} as a single loop  [rank 0]           0       1    1.0000        1.0  True


## Validating Lemma 4.24 (flags of flats via successive minors)

For a loopless matroid $M$ of rank $s$ and a flag $\emptyset = F_0 \subsetneq F_1
\subsetneq \cdots \subsetneq F_k = [d]$, write $N_i = (M|_{F_i})/F_{i-1} = $
`M.minor(F_i, F_{i-1})`. Lemma 4.24(c) claims: *all* $N_i$ are loopless of rank $1$ **iff**
$k = s$ and $\mathcal F$ is a complete flag of flats of $M$ (i.e. every $F_i$ is a flat
with $\operatorname{rk}_M(F_i)=i$).

We check both directions by brute force on `SM({2,4})`, over every flag of subsets of
length $s-1$, $s$, or $s+1$.

In [7]:
def all_chains_of_length(elements, k):
    '''all flags (frozenset(),) -> F_1 -> ... -> F_k = full set, |F_i| strictly increasing'''
    elements = list(elements)
    n = len(elements)
    results = []

    def rec(prev_chain, remaining_sizes):
        if not remaining_sizes:
            results.append(tuple(prev_chain))
            return
        size = remaining_sizes[0]
        prev = prev_chain[-1]
        rest_elems = [e for e in elements if e not in prev]
        need = size - len(prev)
        for extra in itertools.combinations(rest_elems, need):
            rec(prev_chain + [frozenset(prev | set(extra))], remaining_sizes[1:])

    for sizes in itertools.combinations(range(1, n), k - 1):
        rec([frozenset()], list(sizes) + [n])
    return results


def check_flag(M: Matroid, flag) -> tuple:
    k = len(flag) - 1
    minors = [M.minor(flag[i], flag[i - 1]) for i in range(1, k + 1)]
    all_rank1_loopless = all(mi.rank(mi.ground_set) == 1 and len(mi.loops()) == 0 for mi in minors)
    fl = set(M.flats())
    is_complete_flag_of_flats = (k == M.rank(M.ground_set)) and all(
        flag[i] in fl and M.rank(flag[i]) == i for i in range(len(flag))
    )
    return all_rank1_loopless, is_complete_flag_of_flats


M = schubert_matroid(4, frozenset([2, 4]))
gs, rk = sorted(M.ground_set), M.rank(M.ground_set)
tested, mismatches = 0, 0
for k in sorted({max(1, rk - 1), rk, rk + 1}):
    if k > len(gs):
        continue
    for flag in all_chains_of_length(gs, k):
        all_r1l, is_cff = check_flag(M, flag)
        tested += 1
        mismatches += (all_r1l != is_cff)

print(f"SM({{2,4}}) on [4] has rank {rk}.")
print(f"Tested {tested} flags of subsets (lengths {rk-1}, {rk}, {rk+1}); mismatches with Lemma 4.24(c): {mismatches}")
assert mismatches == 0

complete_flags = [flag for flag in all_chains_of_length(gs, rk) if check_flag(M, flag)[1]]
print(f"\nThe {len(complete_flags)} complete flags of flats of SM({{2,4}}) are:")
for flag in complete_flags:
    print("  ", [tuple(sorted(F)) for F in flag])

SM({2,4}) on [4] has rank 2.
Tested 51 flags of subsets (lengths 1, 2, 3); mismatches with Lemma 4.24(c): 0

The 3 complete flags of flats of SM({2,4}) are:
   [(), (1,), (1, 2, 3, 4)]
   [(), (2,), (1, 2, 3, 4)]
   [(), (3, 4), (1, 2, 3, 4)]


## Computing $\Lambda_{\mathcal F}$ for example matroids (Definition 4.25 / Proposition 4.26)

This is the central computational check: we compute the chain functional
$\Lambda_{\mathcal F}(\Psi^-(M))$ two ways —

1. **By definition** (Definition 4.25): sum $\prod_i a(\pi_i)$ over all tuples of set
   compositions $\pi_i \models D_i$ of the pieces $D_i = F_i \setminus F_{i-1}$, weighted by
   whether the concatenation $\pi_1|\cdots|\pi_k$ is $M$-antigeneric.
2. **By the closed form** (Proposition 4.26(1)): the product $\prod_i \delta_{D_i}(N_i)$
   of local rank-one detectors on the successive minors $N_i = $ `M.minor(F_i, F_{i-1})`.

Proposition 4.26 says these must always agree — this is the computational heart of the
whole argument (it is what turns "sum over exponentially many set compositions" into "a
short product of $0/1$ values"), so we check it on three different example matroids and
several flags of subsets of varying length for each.

In [8]:
def concat(setcompositions) -> SetComposition:
    blocks = []
    for sc in setcompositions:
        blocks.extend(sc.parts)
    return SetComposition(blocks)


def Lambda_F_by_definition(M: Matroid, flag) -> float:
    k = len(flag) - 1
    Ds = [sorted(flag[i] - flag[i - 1]) for i in range(1, k + 1)]
    total = 0.0
    for opis in itertools.product(*[all_setcompositions_on(D) for D in Ds]):
        if is_antigeneric(M, concat(opis)):
            coeff = 1.0
            for o in opis:
                coeff *= a_coeff(o)
            total += coeff
    return total


def Lambda_F_closed_form(M: Matroid, flag) -> float:
    k = len(flag) - 1
    prod = 1.0
    for i in range(1, k + 1):
        prod *= delta_S(M.minor(flag[i], flag[i - 1]))
    return prod


examples = [
    ("SM({2,4}) on [4]", schubert_matroid(4, frozenset([2, 4]))),
    ("U(2,3) on [3]", uniform_matroid(3, 2)),
    ("SM({1,3,4}) on [4]", schubert_matroid(4, frozenset([1, 3, 4]))),
]

all_match = True
for name, M in examples:
    gs, rk = sorted(M.ground_set), M.rank(M.ground_set)
    print(f"\n-- {name}: rank {rk}, ground set {gs} --")
    for k in sorted({v for v in (rk - 1, rk, rk + 1) if 1 <= v <= len(gs)}):
        for flag in itertools.islice(all_chains_of_length(gs, k), 3):
            bf = Lambda_F_by_definition(M, flag)
            cf = Lambda_F_closed_form(M, flag)
            match = abs(bf - cf) < 1e-9
            all_match &= match
            flag_str = [tuple(sorted(F)) for F in flag]
            print(f"   k={k}  F={flag_str}   by definition={bf:+.3f}   closed form={cf:+.3f}   match={match}")

assert all_match


-- SM({2,4}) on [4]: rank 2, ground set [1, 2, 3, 4] --
   k=1  F=[(), (1, 2, 3, 4)]   by definition=+0.000   closed form=+0.000   match=True
   k=2  F=[(), (1,), (1, 2, 3, 4)]   by definition=+1.000   closed form=+1.000   match=True
   k=2  F=[(), (2,), (1, 2, 3, 4)]   by definition=+1.000   closed form=+1.000   match=True
   k=2  F=[(), (3,), (1, 2, 3, 4)]   by definition=-0.000   closed form=+0.000   match=True
   k=3  F=[(), (1,), (1, 2), (1, 2, 3, 4)]   by definition=+0.000   closed form=+0.000   match=True
   k=3  F=[(), (1,), (1, 3), (1, 2, 3, 4)]   by definition=+0.000   closed form=+0.000   match=True
   k=3  F=[(), (1,), (1, 4), (1, 2, 3, 4)]   by definition=+0.000   closed form=+0.000   match=True

-- U(2,3) on [3]: rank 2, ground set [1, 2, 3] --
   k=1  F=[(), (1, 2, 3)]   by definition=-0.000   closed form=+0.000   match=True
   k=2  F=[(), (1,), (1, 2, 3)]   by definition=+1.000   closed form=+1.000   match=True
   k=2  F=[(), (2,), (1, 2, 3)]   by definition=+1.000   c

## Hampe's independence theorem (Theorem 4.27)

For loopless nested matroids of a fixed rank $r$ on $[d]$, Hampe's theorem says the
vectors $v_N \in \{0,1\}^{\mathcal C_r}$ — indexed by length-$r$ flags of subsets, and
recording whether every step of the flag is a flat of $N$ — are linearly independent.
This is the external input the main proof of Theorem 4.17 relies on (Section 4.4.7). We
check it for every rank of every $d \le 5$: the rank of the $\{0,1\}$-matrix should equal
the number of loopless nested matroids of that rank (the Eulerian number
$A_{r-1,d}$).

In [9]:
def hampe_vector(N: Matroid, all_length_r_chains) -> list:
    fl = set(N.flats())
    return [1 if all(F in fl for F in chain[1:-1]) else 0 for chain in all_length_r_chains]


for d in range(2, 6):
    nested = generate_loopless_nested_matroids(d)
    by_rank = {}
    for N in nested:
        by_rank.setdefault(N.rank(N.ground_set), []).append(N)

    for r, Ns in sorted(by_rank.items()):
        all_chains = all_chains_of_length(range(1, d + 1), r)
        vectors = [hampe_vector(N, all_chains) for N in Ns]
        matrix_rank = int(np.linalg.matrix_rank(np.array(vectors, dtype=float))) if vectors else 0
        independent = (matrix_rank == len(Ns))
        print(f"d={d} rank={r}: {len(Ns):3d} loopless nested matroids, "
              f"{len(all_chains):4d} length-{r} chains, matrix rank={matrix_rank:3d}  "
              f"independent={independent}")
        assert independent

d=2 rank=1:   1 loopless nested matroids,    1 length-1 chains, matrix rank=  1  independent=True
d=2 rank=2:   1 loopless nested matroids,    2 length-2 chains, matrix rank=  1  independent=True
d=3 rank=1:   1 loopless nested matroids,    1 length-1 chains, matrix rank=  1  independent=True
d=3 rank=2:   4 loopless nested matroids,    6 length-2 chains, matrix rank=  4  independent=True
d=3 rank=3:   1 loopless nested matroids,    6 length-3 chains, matrix rank=  1  independent=True
d=4 rank=1:   1 loopless nested matroids,    1 length-1 chains, matrix rank=  1  independent=True
d=4 rank=2:  11 loopless nested matroids,   14 length-2 chains, matrix rank= 11  independent=True
d=4 rank=3:  11 loopless nested matroids,   36 length-3 chains, matrix rank= 11  independent=True
d=4 rank=4:   1 loopless nested matroids,   24 length-4 chains, matrix rank=  1  independent=True
d=5 rank=1:   1 loopless nested matroids,    1 length-1 chains, matrix rank=  1  independent=True
d=5 rank=2:  26 loop

## The main theorem: $\dim_{\mathbb Q}(\operatorname{im}\Psi)_d = d!$ (Theorem 4.17 / Corollary 4.28)

Finally, the payoff: `z_rank` (already in the package, computing the $\mathbb Q$-dimension
of the span of the WQSym coefficient vectors) applied to *all* loopless nested matroids on
$[d]$ should equal $d!$ exactly — not just the previously-known lower bound $2^d - d$ from
Theorem 4.9. We check $d = 2,\dots,6$; $d=6$ already involves $720$ matroids and around
$4683$ (= Bell$(6)$) monomial-basis coordinates, so it takes a little while.

In [10]:
for d in range(2, 7):
    t0 = time.time()
    nested = generate_loopless_nested_matroids(d)
    rank = z_rank(nested)
    dt = time.time() - t0
    d_factorial = factorial(d)
    lower_bound_old = 2**d - d
    match = (rank == d_factorial == len(nested))
    print(f"d={d}: {len(nested):4d} loopless nested matroids (d! = {d_factorial}), "
          f"z_rank = {rank:4d}, old lower bound 2^d-d = {lower_bound_old:4d}  "
          f"match={match}  [{dt:.1f}s]")
    assert match

d=2:    2 loopless nested matroids (d! = 2), z_rank =    2, old lower bound 2^d-d =    2  match=True  [0.0s]
d=3:    6 loopless nested matroids (d! = 6), z_rank =    6, old lower bound 2^d-d =    5  match=True  [0.0s]
d=4:   24 loopless nested matroids (d! = 24), z_rank =   24, old lower bound 2^d-d =   12  match=True  [0.0s]
d=5:  120 loopless nested matroids (d! = 120), z_rank =  120, old lower bound 2^d-d =   27  match=True  [0.3s]
d=6:  720 loopless nested matroids (d! = 720), z_rank =  720, old lower bound 2^d-d =   58  match=True  [16.7s]


d=6:  720 loopless nested matroids (d! = 720), z_rank =  720, old lower bound 2^d-d =   58  match=True  [18.0s]


## Summary

Every claim checked above matched the theory exactly, on every example tried:

* Lemma 4.20 (exchange posets detect the unique minimum-weight basis) — $0$ mismatches
  out of $375$ (set composition, basis) pairs.
* Lemma 4.22 ($\delta_S$ is the loopless-rank-one indicator) — exact match on $6$ example
  matroids, including the rank-$0$/single-loop edge case.
* Lemma 4.24 (rank-one minors along a flag $\iff$ complete flag of flats) — $0$ mismatches
  out of $51$ flags.
* Proposition 4.26 ($\Lambda_{\mathcal F}$ by definition equals the product of
  $\delta_{D_i}$'s) — matched on $3$ example matroids across several flags each.
* Theorem 4.27 (Hampe's independence) — confirmed for every rank of every $d \le 5$.
* Theorem 4.17 / Corollary 4.28 ($\dim_{\mathbb Q}(\operatorname{im}\Psi)_d = d!$) —
  confirmed for $d = 2, \ldots, 6$.

None of this constitutes a proof for general $d$ (that is what Section 4.4 of the paper
is for) — but it is strong evidence that the AI-drafted argument, as rewritten and
gap-filled in the paper, is correct, and it gives worked examples for a reader trying to
build intuition for an otherwise quite abstract chain of lemmas.

As a side effect of writing the matroid-operation helpers used throughout, we also found
and fixed a real, independent bug in `Matroid.is_nested()` (see the second section above),
now corrected in the package together with the new `flats`, `cyclic_flats`, `loops`,
`restriction`, `contraction` and `minor` methods, with test coverage added in
`package/tests/test_matroids.py`.

As noted at the top of this notebook, this does **not** touch Conjecture 4.14 (the
min-max set composition conjecture tested in `03_min_max_conjecture_testing.ipynb`), which
remains open: the set compositions coming out of a complete flag of flats are generally
*not* the min-max ones.